# Load & Print Results Table (CSV or manual fallback)

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

RESULTS_DIR = 'results'


try:
    df = pd.read_csv(f'{RESULTS_DIR}/results_table.csv')
    print("Results loaded from CSV:")
    print(df.to_string())
    print(f"\nTotal records: {len(df)}")
except FileNotFoundError:
    print("results_table.csv not found!")
    print("\nUsing manual data...")
    df = pd.DataFrame([
        {'dataset': 'MNIST', 'model': 'RecKAN', 'final_test_metric': 0.9720},
        {'dataset': 'MNIST', 'model': 'ChebyKAN', 'final_test_metric': 0.9681},
        {'dataset': 'MNIST', 'model': 'JacobiKAN', 'final_test_metric': 0.9708},
        {'dataset': 'MNIST', 'model': 'SplineKAN', 'final_test_metric': 0.9672},
        {'dataset': 'CIFAR10', 'model': 'RecKAN', 'final_test_metric': 0.5412},
        {'dataset': 'CIFAR10', 'model': 'ChebyKAN', 'final_test_metric': 0.5384},
        {'dataset': 'CIFAR10', 'model': 'JacobiKAN', 'final_test_metric': 0.5311},
        {'dataset': 'CIFAR10', 'model': 'SplineKAN', 'final_test_metric': 0.5130},
        {'dataset': 'AGNews', 'model': 'RecKAN', 'final_test_metric': 0.8846},
        {'dataset': 'AGNews', 'model': 'ChebyKAN', 'final_test_metric': 0.8837},
        {'dataset': 'AGNews', 'model': 'JacobiKAN', 'final_test_metric': 0.8791},
        {'dataset': 'AGNews', 'model': 'SplineKAN', 'final_test_metric': 0.8787},
        {'dataset': 'ETTh1', 'model': 'RecKAN', 'final_test_metric': 0.0135},
        {'dataset': 'ETTh1', 'model': 'ChebyKAN', 'final_test_metric': 0.0372},
        {'dataset': 'ETTh1', 'model': 'JacobiKAN', 'final_test_metric': 0.0390},
        {'dataset': 'ETTh1', 'model': 'SplineKAN', 'final_test_metric': 0.0137},
        {'dataset': 'ECG5000', 'model': 'RecKAN', 'final_test_metric': 0.9418},
        {'dataset': 'ECG5000', 'model': 'ChebyKAN', 'final_test_metric': 0.9376},
        {'dataset': 'ECG5000', 'model': 'JacobiKAN', 'final_test_metric': 0.9384},
        {'dataset': 'ECG5000', 'model': 'SplineKAN', 'final_test_metric': 0.9380},
    ])
    print("Manual data loaded!")

datasets = df['dataset'].unique()
models = ['RecKAN', 'ChebyKAN', 'JacobiKAN', 'SplineKAN']
colors = {'RecKAN': '#d64545', 'ChebyKAN': '#3b6fd6', 'JacobiKAN': '#2ca02c', 'SplineKAN': '#ff7f0e'}

# Detect task type for each dataset
task_types = {}
for ds in datasets:
    ds_data = df[df['dataset'] == ds]
    max_val = ds_data['final_test_metric'].max()
    if max_val > 0.5:
        task_types[ds] = 'classification'
    else:
        task_types[ds] = 'regression'


fig, axes = plt.subplots(1, len(datasets), figsize=(5 * len(datasets), 4))
if len(datasets) == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    ds_data = df[df['dataset'] == ds]
    vals = []
    for m in models:
        try:
            val = ds_data[ds_data['model'] == m]['final_test_metric'].values[0]
            vals.append(val)
        except:
            vals.append(0)
    
    bars = ax.bar(models, vals, color=[colors[m] for m in models])
    
    for bar, m in zip(bars, models):
        if m == 'RecKAN':
            bar.set_edgecolor('black')
            bar.set_linewidth(2)
    
    task = task_types.get(ds, 'classification')
    ylabel = 'Accuracy' if task == 'classification' else 'MSE (lower is better)'
    ax.set_ylabel(ylabel)
    ax.set_title(ds)
    ax.tick_params(axis='x', rotation=15)
    
    for bar, val in zip(bars, vals):
        if val > 0:
            height = bar.get_height()
            offset = 0.01 if task == 'classification' else 0.001
            ax.text(bar.get_x() + bar.get_width()/2., height + offset,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comparison_bar_all_datasets.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/comparison_bar_all_datasets.png")


fig, ax = plt.subplots(figsize=(8, 6))

pivot = df.pivot(index='dataset', columns='model', values='final_test_metric')
pivot_norm = pivot.div(pivot.max(axis=1), axis=0)

im = ax.imshow(pivot_norm.values, cmap='RdYlGn', aspect='auto', vmin=0.8, vmax=1.0)

ax.set_xticks(np.arange(len(pivot_norm.columns)))
ax.set_yticks(np.arange(len(pivot_norm.index)))
ax.set_xticklabels(pivot_norm.columns)
ax.set_yticklabels(pivot_norm.index)

for i in range(len(pivot_norm.index)):
    for j in range(len(pivot_norm.columns)):
        val = pivot_norm.iloc[i, j]
        if not pd.isna(val):
            text = ax.text(j, i, f'{pivot.iloc[i, j]:.3f}',
                          ha='center', va='center', color='black', fontsize=9)

plt.title('Normalized Performance (1.0 = Best per Dataset)')
plt.xlabel('Model')
plt.ylabel('Dataset')
plt.colorbar(im, label='Relative Performance')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/performance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/performance_heatmap.png")

print('\n' + '=' * 80)
print('FINAL RESULTS SUMMARY')
print('=' * 80)

winners = {}
for ds in datasets:
    ds_data = df[df['dataset'] == ds]
    if task_types.get(ds, 'classification') == 'classification':
        winner = ds_data.loc[ds_data['final_test_metric'].idxmax()]
    else:
        winner = ds_data.loc[ds_data['final_test_metric'].idxmin()]
    winners[ds] = winner['model']

win_counts = {}
for m in models:
    win_counts[m] = list(winners.values()).count(m)

for ds in datasets:
    ds_data = df[df['dataset'] == ds]
    task = task_types.get(ds, 'classification')
    print(f"\n{ds} ({task}):")
    for _, row in ds_data.iterrows():
        marker = ' [WINNER]' if row['model'] == winners[ds] else ''
        print(f"  {row['model']:12s}: {row['final_test_metric']:.4f}{marker}")
    print(f"  Winner: {winners[ds]}")

print('\n' + '-' * 80)
print('WINNER COUNT:')
for m in models:
    count = win_counts.get(m, 0)
    print(f"  {m:12s}: {count} dataset(s)")
print('=' * 80)
print(f"All plots saved in {RESULTS_DIR}/")
print('=' * 80)